In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Data preparation

In [2]:
#data preparation
df=pd.read_csv('./WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.columns = df.columns.str.lower().str.replace(' ', '_')
catergorical_columns= list(df.dtypes[df.dtypes == 'str'].index)
for col in catergorical_columns:
    df[col]=df[col].str.lower().str.replace(' ','_')
#upon inspection, totalcharges column should be numeric. But it is Str

In [3]:
df[df.totalcharges == '_'][['customerid', 'totalcharges']]

,customerid,totalcharges
488,4472-lvygi,_
753,3115-czmzd,_
936,5709-lvoeq,_
1082,4367-nuyao,_
1340,1371-dwpaz,_
3331,7644-omvmy,_
3826,3213-vvolg,_
4380,2520-sgtta,_
5218,2923-arzlg,_
6670,4075-wkniu,_


In [4]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce') # '_' values are converted to NaN
df.totalcharges =  df.totalcharges.fillna(0)

In [5]:
df.churn # in ML classification it should be 1 or 0
df.churn = (df.churn == 'yes').astype(int)

# Validation framework

In [6]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val=train_test_split(df_full_train, test_size=0.25, random_state=1 ) #25% of df_full_train is 20% of original df

In [7]:
df_train=df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)
df_full_train=df_full_train.reset_index(drop=True)

In [8]:
y_train=df_train.churn.values
y_val=df_val.churn.values
y_test=df_test.churn.values

del df_train['churn']
del df_val['churn']
del df_test['churn']

# EDA

In [9]:
df_full_train.isnull().sum()
#no missing values

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [10]:
df_full_train.churn.value_counts(normalize=True)
#churn rate. As it is binary, churn rate can be calculated as mean also

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [11]:
numerical=['tenure','monthlycharges','totalcharges']
categorical=['gender', 'seniorcitizen', 'partner', 'dependents',
        'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod']

# numerical and customer id and churn are removed from categorical.

In [21]:
df_full_train[categorical].nunique()
#most of the variables are not binary

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

In [13]:
df_full_train['contract'].value_counts()

contract
month-to-month    3104
two_year          1344
one_year          1186
Name: count, dtype: int64

# Feature importance and risk

In [14]:
#churn rate
churn_female=df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_male=df_full_train[df_full_train.gender == 'male'].churn.mean()
global_churn=df_full_train.churn.mean()

In [15]:
print(churn_female, churn_male, global_churn)
#nearly similar

0.27682403433476394 0.2632135306553911 0.26996805111821087


In [16]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
# churn rate of people with partner is less
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()

In [17]:
global_churn - churn_partner
# 6% difference in churn of people with partner
#partner is an important variable than gender

np.float64(0.06493474245795922)

In [18]:
# risk ratio 
churn_no_partner / global_churn

np.float64(1.2216593879412643)

In [19]:
# risk ratio 
churn_partner / global_churn

np.float64(0.7594724924338315)

In [22]:
from IPython.display import display

for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['risk'] = df_group['mean'] / global_churn
    display(df_group)
    print()
    print()

gender


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980




seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208




partner


...

,mean,count,diff,risk
contract,,,,
month-to-month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730




paperlessbilling


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560




paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121
